### Import Libraries

In [18]:
import pandas as pd
import plotly.express as px

from dash import Dash, dcc, html, Input, Output
from dash import Dash

### Loading Dataset

In [20]:
df = pd.read_csv("netflix_titles.csv")

df.head(2)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."


### Data Cleaning

In [22]:
# Fill missing values only in text columns

text_columns = [
    "director",
    "cast",
    "country",
    "rating",
    "duration"
]


for col in text_columns:
    df[col] = df[col].fillna("Unknown")


# Fill numeric column separately

df["release_year"] = df["release_year"].fillna(
    df["release_year"].median()
)


# Convert date

df["date_added"] = pd.to_datetime(
    df["date_added"],
    errors="coerce"
)


# Extract year

df["year_added"] = df["date_added"].dt.year

### Creating Dashboard

In [28]:
app = Dash(__name__)


app.layout = html.Div([


    html.H1(
        "Netflix Titles Analysis Dashboard",
        style={
            "textAlign":"center"
        }
    ),



    html.Div([


        html.Div([
            html.H3("Total Titles"),
            html.H2(id="total")
        ]),



        html.Div([
            html.H3("Movies"),
            html.H2(id="movies")
        ]),



        html.Div([
            html.H3("TV Shows"),
            html.H2(id="shows")
        ])


    ],
    style={
        "display":"flex",
        "justifyContent":"space-around"
    }),



    html.Br(),


    html.H3("Select Content Type"),


    dcc.Dropdown(

        id="type_filter",

        options=[

            {
                "label":x,
                "value":x
            }

            for x in df["type"].unique()

        ],

        value="Movie"

    ),



    dcc.Graph(
        id="trend"
    ),



    dcc.Graph(
        id="country"
    ),



    dcc.Graph(
        id="rating"
    )


])

In [30]:
@app.callback(

[

Output("total","children"),

Output("movies","children"),

Output("shows","children"),

Output("trend","figure"),

Output("country","figure"),

Output("rating","figure")

],


Input(
"type_filter",
"value"
)

)



def update_dashboard(selected_type):


    filtered = df[
        df["type"] == selected_type
    ]



    # KPI values

    total_titles = len(df)


    movies = len(
        df[df["type"]=="Movie"]
    )


    shows = len(
        df[df["type"]=="TV Show"]
    )



    # -------- Trend Chart --------


    trend_data = (

        filtered

        .dropna(
            subset=["year_added"]
        )

        .groupby(
            "year_added"
        )

        .size()

        .reset_index(
            name="count"
        )

    )


    trend_fig = px.line(

        trend_data,

        x="year_added",

        y="count",

        title="Content Added Over Years"

    )




    # -------- Country Chart --------


    country_data = (

        filtered["country"]

        .value_counts()

        .head(10)

        .reset_index()

    )


    country_data.columns = [

        "Country",

        "Count"

    ]


    country_fig = px.bar(

        country_data,

        x="Country",

        y="Count",

        title="Top 10 Countries"

    )





    # -------- Rating Chart --------


    rating_data = (

        filtered["rating"]

        .value_counts()

        .reset_index()

    )


    rating_data.columns = [

        "Rating",

        "Count"

    ]



    rating_fig = px.pie(

        rating_data,

        names="Rating",

        values="Count",

        title="Rating Distribution"

    )



    return (

        total_titles,

        movies,

        shows,

        trend_fig,

        country_fig,

        rating_fig

    )

In [32]:
app.run(
    jupyter_mode="inline",
    port=8050
)